aipw estimators

In [ ]:
#%pip install openpyxl
#%pip install io
#%pip install datetime
#%pip install econtools
#%pip install geopy.distance
#%pip install pyreadstat
#%pip install pyarrow
#%pip install scipy
#%pip install pyreadr
#%pip install scipy.stats
#%pip install statsmodels
#%pip install matplotlib.pyplot
#%pip install seaborn
#%pip install os


In [ ]:
#import dbm.sqlite3
import openpyxl
import pandas as pd
import numpy as np
from io import StringIO
import datetime as dt
import econtools
import geopy.distance as geo
import pyreadstat
import pyarrow
import scipy
import pyreadr
import scipy.stats as stats
import statsmodels
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
import os, shutil
from pathlib import Path
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

#use a list
packages = ['math','random','xlrd']
modules = map(__import__,packages)

from datetime import date
print("Today's date is:",date.today())

In [ ]:
#Pandas version is 3.0.2. This is the latest version.
import pandas as pd
print(f"Pandas version: {pd.__version__}")

In [ ]:
### 

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from statsmodels.formula.api import glm
from sklearn.model_selection import train_test_split


# Load the data
# Step 1: Set the working directory (No direct equivalent, specify path)
data_path = "/Causal_Inference/Treatment_effects/data/cattaneo2.dta"
# Load the data (use pandas for this)
df = pd.read_stata(data_path,convert_categoricals=False)
# Print first few rows of the data
print(df.head())

# Step 2: Rename variables (using pandas)
df = df.rename(columns={'mbsmoke': 'treat'})
# Display the first few rows of the data
print(df.head())

# Step 2: Estimate the propensity score using logistic regression
# Logistic regression model to estimate the propensity score
X = df[['mmarried', 'mage', 'fbaby', 'medu']]  # Covariates
y = df['treat']  # Treatment variable

# Fit logistic regression model
logit_model = LogisticRegression()
logit_model.fit(X, y)

# Calculate the propensity scores (probabilities of treatment)
df['pscore'] = logit_model.predict_proba(X)[:, 1]

# Step 3: Compute the inverse probability weights (IPW) for treated and control groups
df['ipw0'] = np.where(df['treat'] == 0, 1 / (1 - df['pscore']), 0)  # IPW for control group
df['ipw1'] = np.where(df['treat'] == 1, 1 / df['pscore'], 0)  # IPW for treated group

# Step 4: Estimate the Potential Outcomes Model (POM) for treated group
# No weights in the outcome model for the treated group
treated_data = df[df['treat'] == 1]

# Fit the regression model for the treated group
model_treat = sm.WLS(treated_data['bweight'], sm.add_constant(treated_data[['mage', 'prenatal1', 'mmarried', 'fbaby']]), 
                     weights=treated_data['ipw1']).fit()

# Generate potential outcome for treated group (POM1) using specified coefficients
df['pom1'] = 3227.169 + (41.43991 * df['fbaby']) + (133.6617 * df['mmarried']) + \
               (25.11133 * df['prenatal1']) + (-7.370881 * df['mage'])

# Adjust the prediction with the inverse probability weight (IPW) for treated group
df['pom1'] = df['pom1'] + df['ipw1'] * (df['bweight'] - df['pom1'])

# Step 5: Estimate the Potential Outcomes Model (POM) for control group
control_data = df[df['treat'] == 0]

# Fit the regression model for the control group
model_control = sm.WLS(control_data['bweight'], 
                       sm.add_constant(control_data[['mage', 'prenatal1', 'mmarried', 'fbaby']]), 
                       weights=control_data['ipw0']).fit()

# Generate potential outcome for control group (POM0) using specified coefficients
df['pom0'] = 3202.746 + (-71.3286 * df['fbaby']) + (160.9513 * df['mmarried']) + \
               (64.40859 * df['prenatal1']) + (2.546828 * df['mage'])

# Adjust the prediction with the inverse probability weight (IPW) for control group
df['pom0'] = df['pom0'] + df['ipw0'] * (df['bweight'] - df['pom0'])

# Step 6: Calculate the average treatment effect (ATE) using the potential outcomes
pom_c = df['pom0'].mean()  # Mean potential outcome for control group
pom_t = df['pom1'].mean()  # Mean potential outcome for treated group

# Calculate the ATE (Average Treatment Effect)
ATE = pom_t - pom_c

# Print the ATE
print(f"AIPW = {ATE}")


In [ ]:
import datetime
print(f"aipw estimator finished")
print(f"Program completed on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")